In [0]:
table_name = "classic_stable_paco_catalog.ws_rico_martinez.sap_hana_pedidos_demo"

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name} AS
WITH base AS (
  SELECT
    id,
    date_sub(current_date(), CAST(rand(7) * 180 AS INT)) AS order_date,
    CAST(rand(11) * 8 + 1 AS INT) AS quantity,
    ROUND(rand(13) * 750 + 25, 2) AS unit_price,
    ROUND(rand(17) * 0.18, 4) AS discount_pct,
    rand(19) AS return_prob,
    rand(23) AS status_prob,
    rand(29) AS risk_prob
  FROM range(25000)
)
SELECT
  CAST(id + 100000 AS BIGINT) AS order_id,
  CONCAT('SAPC', LPAD(CAST((id % 6000) + 1 AS STRING), 6, '0')) AS customer_id,
  CONCAT('Cliente ', LPAD(CAST((id % 6000) + 1 AS STRING), 6, '0')) AS customer_name,
  element_at(array('Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Bucaramanga', 'Cartagena', 'Pereira', 'Manizales'), CAST((id % 8) + 1 AS INT)) AS city,
  element_at(array('Centro', 'Retail', 'Mayorista', 'Digital', 'Gobierno'), CAST((id % 5) + 1 AS INT)) AS sales_channel,
  element_at(array('Tecnología', 'Hogar', 'Salud', 'Moda', 'Alimentos', 'Ferretería'), CAST((id % 6) + 1 AS INT)) AS product_family,
  element_at(array('Básico', 'Estándar', 'Premium'), CAST((id % 3) + 1 AS INT)) AS customer_tier,
  order_date,
  date_add(order_date, CAST(rand(31) * 12 + 1 AS INT)) AS delivery_date,
  quantity,
  unit_price,
  discount_pct,
  ROUND(quantity * unit_price, 2) AS gross_amount,
  ROUND(quantity * unit_price * (1 - discount_pct), 2) AS net_amount,
  ROUND(quantity * unit_price * (1 - discount_pct) * 0.19, 2) AS vat_amount,
  ROUND(quantity * unit_price * (1 - discount_pct) * 1.19, 2) AS total_amount,
  element_at(array('Transferencia', 'Tarjeta', 'Crédito', 'Débito'), CAST((id % 4) + 1 AS INT)) AS payment_method,
  CASE
    WHEN risk_prob < 0.08 THEN 'ALTO'
    WHEN risk_prob < 0.22 THEN 'MEDIO'
    ELSE 'BAJO'
  END AS credit_risk,
  CASE
    WHEN status_prob < 0.07 THEN 'CANCELADO'
    WHEN status_prob < 0.18 THEN 'DEVUELTO'
    WHEN status_prob < 0.55 THEN 'DESPACHADO'
    ELSE 'ENTREGADO'
  END AS order_status,
  CASE WHEN return_prob < 0.09 THEN 'SI' ELSE 'NO' END AS return_flag,
  current_timestamp() AS last_updated_ts,
  'SAP_HANA_DEMO' AS source_system
FROM base
""")

summary = spark.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  ROUND(SUM(total_amount), 2) AS total_sales
FROM {table_name}
""")

preview = spark.sql(f"""
SELECT *
FROM {table_name}
ORDER BY order_id
LIMIT 10
""")

print(f"Tabla creada: {table_name}")
display(summary)
display(preview)


## Integración SAP HANA ↔ Databricks — Flujo del Taller

Este notebook demuestra el ciclo completo de integración bidireccional entre SAP HANA y Databricks.

| Paso | Acción | Capacidad Databricks |
|------|--------|----------------------|
| **1** | Leer tabla `PEDIDOS_DEMO` desde SAP HANA | JDBC / Databricks Connection |
| **2** | Enriquecer y calcular KPIs | Spark + Window Functions |
| **3** | Escribir `PEDIDOS_ENRIQUECIDOS` de vuelta a SAP HANA | JDBC write paralelo |

> **Pre-requisito:** exportar como CSV la tabla generada en la celda inicial y cargarla al esquema `WORKSHOP` en SAP HANA antes del paso 1.

## 1. Leer datos desde SAP HANA via JDBC

Usando la **Databricks Connection** registrada en Unity Catalog, se accede a la tabla `PEDIDOS_DEMO` directamente desde SAP BW vía JDBC — sin credenciales en el notebook.

> La conexión es configurada por el administrador una sola vez en **Catalog Explorer → External Data → Connections**.  
> Los usuarios solo necesitan permisos de uso sobre la conexión para ejecutar esta celda.

In [0]:
%sql
-- Conexiones registradas en Unity Catalog
-- Catalog Explorer → External Data → Connections
-- Identificar aquí el nombre de la conexión SAP HANA para usarlo en la celda siguiente
SHOW CONNECTIONS

In [0]:
# ─── Paso 1: Leer desde SAP BW usando Databricks Connection ──────────────────
# La conexión fue registrada en: Catalog Explorer → External Data → Connections
# Se referencia directamente por nombre — sin catálogo externo ni credenciales.

CONNECTION_NAME = "sap_bw_workshop"    # ← Nombre de la conexión BW del taller
SAP_SCHEMA      = "WORKSHOP"           # ← Esquema SAP BW donde está la tabla
SAP_TABLE_IN    = "PEDIDOS_DEMO"
SAP_TABLE_OUT   = "PEDIDOS_ENRIQUECIDOS"

# ─── Leer usando la conexión registrada — sin URL ni credenciales en el código ───
pedidos_raw = (
    spark.read.format("jdbc")
    .option("connectionName",  CONNECTION_NAME)
    .option("dbtable",         f'"{SAP_SCHEMA}"."{SAP_TABLE_IN}"')
    .option("numPartitions",   8)
    .option("partitionColumn", "ORDER_ID")
    .option("lowerBound",      100000)
    .option("upperBound",      200010)
    .option("fetchsize",       5000)
    .load()
)

# SAP BW retorna columnas en MAYÚSCULAS — normalizar a minúsculas
pedidos_raw = pedidos_raw.toDF(*[c.lower() for c in pedidos_raw.columns])

print(f"✔ Conexión  : {CONNECTION_NAME}")
print(f"✔ Fuente    : {SAP_SCHEMA}.{SAP_TABLE_IN}")
print(f"✔ Columnas  : {len(pedidos_raw.columns)}")
print(f"✔ Registros : {pedidos_raw.count():,}")
display(pedidos_raw.limit(5))


### Desglose línea a línea — `pedidos_raw`

| Línea | Qué hace |
|-------|----------|
| `spark.read.format("jdbc")` | Le indica a Spark que usará el conector JDBC para leer desde una base de datos relacional externa. |
| `.option("connectionName", CONNECTION_NAME)` | Referencia la conexión registrada en Unity Catalog por nombre. Spark obtiene la URL, usuario y contraseña de forma segura en tiempo de ejecución — sin exponerlos en el código. |
| `.option("dbtable", f'"{SAP_SCHEMA}"."{SAP_TABLE_IN}"')` | Tabla a leer completamente calificada con su esquema. Las comillas dobles son sintaxis SAP BW/HANA para preservar el caso exacto del nombre. |
| `.option("numPartitions", 8)` | Divide la lectura en 8 tareas paralelas. 8 workers de Spark obtienen datos simultáneamente en lugar de un solo hilo secuencial. |
| `.option("partitionColumn", "ORDER_ID")` | Columna usada para dividir los datos entre las 8 particiones. Debe ser numérica y con distribución relativamente uniforme. |
| `.option("lowerBound", 100000)` / `.option("upperBound", 200010)` | Rango que Spark divide en 8 intervalos iguales (~12 500 filas por partición). **No filtran filas** — registros fuera del rango también se incluyen; sólo controlan cómo se reparte el trabajo. |
| `.option("fetchsize", 5000)` | Número de filas que cada partición trae por viaje de red hacia SAP BW. Valores mayores reducen los round-trips pero consumen más memoria del driver. |
| `.load()` | Dispara la lectura y retorna un DataFrame Spark (**lazy** — SAP BW no se consulta realmente hasta que se ejecute una acción como `.count()` o `display()`). |
| `pedidos_raw.toDF(*[c.lower() …])` | SAP BW devuelve los nombres de columna en `MAYÚSCULAS`. Esta línea los renombra todos a `minúsculas` para que el resto del notebook funcione sin errores de case. |

## 2. Transformar y enriquecer en Databricks

Aplicamos capacidades nativas de Databricks sobre los datos traídos desde SAP HANA:

* **Window functions** — métricas acumuladas por cliente (`total_cust_spend`, `avg_order_value`) sin GROUP BY
* **Segmentación de clientes** — VIP / PREMIUM / EN_RIESGO / REGULAR basada en gasto histórico y riesgo
* **KPIs de entrega** — días de entrega y cumplimiento de SLA (≤ 7 días = CUMPLIDO)
* **KPIs financieros** — tasa de realización de precio (`net / gross`)
* **Trazabilidad** — columnas `processed_by` y `processed_at` para auditoría

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Ventana por cliente — agrega métricas sin reducir filas
w_cust = Window.partitionBy("customer_id")

pedidos_enriquecido = (
    pedidos_raw

    # ── KPIs de entrega ───────────────────────────────────────────────────────
    .withColumn("delivery_days",
        F.datediff(F.col("delivery_date"), F.col("order_date")))
    .withColumn("delivery_sla",
        F.when(F.col("delivery_days") <= 7, "CUMPLIDO").otherwise("DEMORADO"))

    # ── KPIs financieros ──────────────────────────────────────────────────────
    .withColumn("price_realization",
        F.round(F.col("net_amount") / F.col("gross_amount"), 4))
    .withColumn("order_month",
        F.date_format("order_date", "yyyy-MM"))

    # ── Métricas por cliente (window functions) ───────────────────────────────
    .withColumn("total_cust_orders",
        F.count("order_id").over(w_cust))
    .withColumn("total_cust_spend",
        F.round(F.sum("total_amount").over(w_cust), 2))
    .withColumn("avg_order_value",
        F.round(F.avg("total_amount").over(w_cust), 2))

    # ── Segmentación de clientes ──────────────────────────────────────────────
    .withColumn("customer_segment",
        F.when((F.col("total_cust_spend") > 15000) & (F.col("credit_risk") == "BAJO"), "VIP")
         .when( F.col("total_cust_spend") > 8000,                                       "PREMIUM")
         .when( F.col("credit_risk")      == "ALTO",                                    "EN_RIESGO")
         .otherwise("REGULAR"))

    # ── Trazabilidad ──────────────────────────────────────────────────────────
    .withColumn("processed_by", F.lit("DATABRICKS"))
    .withColumn("processed_at", F.current_timestamp())
)

print(f"✔ Registros enriquecidos : {pedidos_enriquecido.count():,}")

# Distribucion de segmentos
display(
    pedidos_enriquecido
    .groupBy("customer_segment", "delivery_sla")
    .agg(
        F.count("order_id").alias("num_ordenes"),
        F.round(F.sum("total_amount"), 2).alias("venta_total"),
        F.round(F.avg("delivery_days"), 1).alias("dias_entrega_prom")
    )
    .orderBy("customer_segment", "delivery_sla")
)


## 3. Escribir resultado de vuelta a SAP HANA

El DataFrame enriquecido se escribe en la nueva tabla `PEDIDOS_ENRIQUECIDOS` en SAP BW usando la misma **Databricks Connection** — sin credenciales en el notebook.

* **`overwrite`** — reemplaza la tabla destino si ya existe (útil para re-ejecuciones del taller)
* **`append`** — acumula nuevos registros sin borrar los existentes (modo producción)
* `batchsize = 10 000` optimiza el throughput de escritura a SAP BW

In [0]:
# ─── Paso 3: Escribir tabla enriquecida de vuelta a SAP BW ──────────────────
# Usa la misma conexión registrada en UC — sin credenciales en el notebook.
(
    pedidos_enriquecido
    .write
    .format("jdbc")
    .option("connectionName", CONNECTION_NAME)
    .option("dbtable",        f'"{SAP_SCHEMA}"."{SAP_TABLE_OUT}"')
    .option("batchsize",      10000)   # Registros por batch
    .option("numPartitions",  4)       # Conexiones paralelas de escritura
    .mode("overwrite")                 # Cambiar a 'append' en producción
    .save()
)

rows_sent = pedidos_enriquecido.count()
print(f"✔ Escritura completada  → {SAP_SCHEMA}.{SAP_TABLE_OUT}")
print(f"✔ Total registros enviados a SAP BW: {rows_sent:,}")


### Desglose línea a línea — write-back a SAP BW

| Línea | Qué hace |
|-------|----------|
| `pedidos_enriquecido.write` | Abre el modo escritura sobre el DataFrame enriquecido. A diferencia de las transformaciones (lazy), a partir de aquí Spark prepara la ejecución real. |
| `.format("jdbc")` | Usa el conector JDBC para escribir en una base de datos relacional externa — el mismo formato que se usó para leer. |
| `.option("connectionName", CONNECTION_NAME)` | Reutiliza la misma conexión registrada en Unity Catalog. No se necesitan credenciales adicionales para el write-back. |
| `.option("dbtable", f'"{SAP_SCHEMA}"."{SAP_TABLE_OUT}"')` | Tabla destino en SAP BW. Si no existe, SAP BW la crea automáticamente con el esquema inferido del DataFrame. |
| `.option("batchsize", 10000)` | Inserta 10 000 filas por cada llamada al driver JDBC. Reduce la cantidad de round-trips y mejora el throughput de escritura. |
| `.option("numPartitions", 4)` | Abre 4 conexiones paralelas hacia SAP BW para escribir simultáneamente. Más particiones = mayor velocidad, pero más carga en SAP BW. |
| `.mode("overwrite")` | Reemplaza la tabla destino completa si ya existe. Cambiar a `"append"` para acumular registros sin borrar los anteriores (modo producción). |
| `.save()` | **Dispara la escritura** — este sí es eager, no lazy. Spark ejecuta todo el plan (transformaciones + write) en este momento y bloquea hasta completar. |
| `pedidos_enriquecido.count()` | Cuenta las filas escritas para confirmar el resultado. Ejecuta una acción adicional sobre el DataFrame (SAP BW ya fue actualizado en el paso anterior). |